In [3]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, f1_score
from imblearn.under_sampling import RandomUnderSampler

# Create models directory
os.makedirs('models', exist_ok=True)

# ==========================================
# PART 1: TRAIN THE MULTI-DISEASE SYSTEM (BRFSS)
# ==========================================
print("--- Loading 1 Million Row Dataset ---")
df = pd.read_csv('./data/processed/multi_year_health_data.csv')

# Define Features (Inputs) and Targets (Diseases)
# Note: We exclude 'Year' and 'State' to make the model generalizable
X_columns = ['Age', 'BMI', 'Smoker', 'PhysicalHealthDays', 'MentalHealthDays', 
             'Income', 'Education', 'Avg_AQI']

# List of diseases to predict
targets = ['HeartAttack', 'Angina', 'Stroke', 'Asthma', 'SkinCancer', 
           'OtherCancer', 'COPD', 'Depression', 'KidneyDisease', 
           'Diabetes', 'Arthritis', 'DifficultyWalking']

print(f"Training models for {len(targets)} diseases...")

model_performance = {}

for target in targets:
    print(f"\n[Training Module] Target: {target}")
    
    # 1. Prepare Data for this specific disease
    # Remove rows where target is missing (if any)
    df_clean = df.dropna(subset=X_columns + [target])
    
    X = df_clean[X_columns]
    y = df_clean[target]
    
    # 2. Handle Imbalance (Crucial for Health Data)
    # Most people don't have the disease. We downsample the healthy class to 1:1 ratio
    # This makes training SUPER FAST and the model more sensitive to risk.
    rus = RandomUnderSampler(random_state=42)
    X_res, y_res = rus.fit_resample(X, y)
    
    # 3. Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42)
    
    # 4. Baseline Model (Logistic Regression) - Requirement: "Baseline Comparison"
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train, y_train)
    y_pred_base = log_reg.predict(X_test)
    base_acc = accuracy_score(y_test, y_pred_base)
    
    # 5. Advanced Model (Random Forest) - Requirement: "Advanced Model"
    rf_model = RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42)
    rf_model.fit(X_train, y_train)
    y_pred_adv = rf_model.predict(X_test)
    adv_acc = accuracy_score(y_test, y_pred_adv)
    adv_f1 = f1_score(y_test, y_pred_adv)
    
    print(f"   -> Baseline Accuracy: {base_acc:.2%}")
    print(f"   -> Random Forest Accuracy: {adv_acc:.2%} (F1: {adv_f1:.2f})")
    
    # 6. Save the Best Model
    joblib.dump(rf_model, f'models/model_{target}.pkl')
    model_performance[target] = adv_acc

print("\n--- BRFSS Training Complete. Models Saved. ---")


# ==========================================
# PART 2: TRAIN SPECIALIZED MODULES
# ==========================================

# --- A. Parkinson's (Regression) ---
print("\n[Training Module] Parkinson's Disease (Voice Analysis)")
try:
    df_pk = pd.read_csv('data/processed/parkinsons_clean.csv')
    # Predict 'total_UPDRS' (Parkinson's score) based on voice features
    # Drop non-feature columns
    drop_cols = ['subject#', 'age', 'sex', 'test_time', 'motor_UPDRS', 'total_UPDRS']
    X_pk = df_pk.drop(columns=[c for c in drop_cols if c in df_pk.columns]) 
    y_pk = df_pk['total_UPDRS']
    
    X_train_pk, X_test_pk, y_train_pk, y_test_pk = train_test_split(X_pk, y_pk, test_size=0.2)
    
    pk_model = RandomForestRegressor(n_estimators=50, max_depth=10)
    pk_model.fit(X_train_pk, y_train_pk)
    print(f"   -> Model Trained. R2 Score: {pk_model.score(X_test_pk, y_test_pk):.2f}")
    
    joblib.dump(pk_model, 'models/model_parkinsons.pkl')
except Exception as e:
    print(f"   -> Skipped Parkinson's: {e}")

# --- B. Autism (Classification) ---
print("\n[Training Module] Autism Screening")
try:
    df_aut = pd.read_csv('data/processed/autism_clean.csv')
    # Assuming last col is target 'ASD_Class' and others are features
    # We need to encode categorical features if any
    X_aut = df_aut.drop(columns=['ASD_Class'])
    # Simple one-hot encoding for categorical text data
    X_aut = pd.get_dummies(X_aut, drop_first=True)
    y_aut = df_aut['ASD_Class']
    
    X_train_aut, X_test_aut, y_train_aut, y_test_aut = train_test_split(X_aut, y_aut, test_size=0.2)
    
    aut_model = RandomForestClassifier(n_estimators=50)
    aut_model.fit(X_train_aut, y_train_aut)
    print(f"   -> Model Trained. Accuracy: {accuracy_score(y_test_aut, aut_model.predict(X_test_aut)):.2%}")
    
    joblib.dump(aut_model, 'models/model_autism.pkl')
    # Save the columns to ensure UI matches training shape
    joblib.dump(X_aut.columns.tolist(), 'models/autism_features.pkl')
    
except Exception as e:
    print(f"   -> Skipped Autism: {e}")

print("\nAll Systems Go! Ready for Streamlit UI.")

--- Loading 1 Million Row Dataset ---


FileNotFoundError: [Errno 2] No such file or directory: './data/processed/multi_year_health_data.csv'